In [98]:
# NAAnoBot training prototype

In [1]:
import torch
import numpy as np
import pandas as pd
from torch.nn.functional import mse_loss
%matplotlib inline
from pathlib import Path

In [2]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

RADIAL_SEQUENCES = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
RADIAL_SEQUENCES.set_index("PDB_ID", inplace=True)

MOLECULAR_FINGERPRINTS = pd.read_pickle(DATAPATH / "molfp_df.pkl")
MOLECULAR_FINGERPRINTS.set_index("smiles_str", inplace=True)

train_pointers_path = DATAPATH / "naanobot_training_pointers_dict.parquet"
test_pointers_path = DATAPATH / "naanobot_test_pointers_dict.parquet"

In [3]:
from src.nano_maker.modules.nAAno.naanoeng import NAAnoEng

A -- [9.0, 0, 6.02, 4.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14.6, 8.7, 7.1, 10.0]
C -- [12.0, 0, 5.02, 4.9, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 9.3, 13.700000000000001, 8.4, 9.7]
D -- [13.0, -5, 2.98, -5.5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9.6, 5.4, 13.100000000000001, 8.7]
E -- [15.0, -5, 3.22, -3.1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14.6, 6.5, 8.5, 8.9]
F -- [17.0, 0, 5.48, 10, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 10.4, 14.399999999999999, 7.0, 8.8]
G -- [8.0, 0, 5.97, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4.2, 3.5999999999999996, 18.5, 9.6]
H -- [16.0, 0, 7.59, 0.8, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8.5, 11.6, 10.2, 8.9]
I -- [13.0, 0, 5.98, 10.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 8.6, 18.5, 5.8999999999999995, 7.6]
K -- [15.0, 5, 9.47, -2.3000000000000003, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11.200000000000001, 8.6, 9.9, 12.4]
L -- [13.0, 0, 5.98, 9.7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 13.200000000000001, 11.5, 6.5, 7.6]
M -- [15.0, 0, 5.

In [4]:
# protein feature engineering:
# spatial data + physicochemical data
# add parts of GEM to append a physiochemical vector on top of spatial one
# - charge, hydrophobicity, pKa, H-bond capacity, .etc
# GEM properties = nAAno_token

# feature vectors as "tokens" per amino acid
# put stuff in nAAno_library for physicochemical analysis
from src.nano_maker.modules.nAAno.naanolibrary import *
import torch

class NAAnoEng:
    """Run this everytime we need a new set of feature vectors"""
    def __init__(self, max_angstroms, block_size, verbose=False):
        self.max_angstroms = max_angstroms
        self.block_size = block_size
        self.verbose = verbose

    def initialize(self):
        self.nAAno_vectors = {aa_id: self.build_nAAnoVector(aa_id) for aa_id in AA_IDS}

        if self.verbose:
            print("NAAnoEng initialized")
        return True

    def n_features(self):
        return len(self.nAAno_vectors['A'])

    def get_aa_id(self, naano_vector):
        aa_id = None
        for code, key in self.nAAno_vectors.items():
            if key == naano_vector:
                aa_id = code
                break

        if aa_id is None:
            raise ValueError(f"Feature vector {naano_vector} presented does not exist")

        return aa_id

    def get_nAAnovector(self, aa_id):
        return self.nAAno_vectors[aa_id]

    @staticmethod
    def build_nAAnoVector(aa_id: str):
        """
        use this in these two scenarios:
        \n    - generating embeddings after updating feature vector
        \n    - inference
        :param aa_id: single letter amino acid reference code
        :returns: feature vector representing a given (valid) amino acid
        """
        # sanity check
        if aa_id not in AA_IDS:
            raise ValueError(f"{aa_id} not in valid AA ID list")

        # embedding scheme, MAKE SURE TO UPDATE THIS IF YOU EVER UPDATE NAANOLIBRARY
        naano_vector = [
            MOLECULAR_WEIGHTS[aa_id] * 100,  # scale up molecular weights by a lot, all in 0.1 range
            NET_CHARGES[aa_id] * 5,  # scale up charge
            ISOELECTRIC_PTS[aa_id],
            HYDROPHOBICITY_IDXS[aa_id] * 10, # same thing
            # generally we want all of them to be on a whole number ish scale
        ]
        naano_vector += FUNCTIONAL_FP[aa_id]
        naano_vector += [p * 10 for p in PROPENSITIES[aa_id]]
        return naano_vector

    # generation + training data processing
    def get_nAAno_X(self, coord_context, bioch_context, coord_Y):
        """
        Generates contextual data for naano dataset class and NAAnoBot's amino acid cage generation
        :param coord_context:
        :param bioch_context:
        :param coord_Y:
        :return:
        """
        # go through coordinate context (iterate through range) <- vectorized now
        # calculate and add relative coordinates concat. with nAAnovectors
        # construct augmented relative coordinate vector then concat with nAAno_token
        coord_X_tensor = torch.tensor(coord_context, dtype=torch.float32)
        bioch_tensor = torch.tensor(bioch_context, dtype=torch.float32)
        coord_Y_tensor = torch.tensor(coord_Y, dtype=torch.float32)

        # XYZ
        xyz_X = self.sph_to_xyz(coord_X_tensor)
        # workaround the batch processing nature -> add dimension at 0th then undo
        xyz_Y = self.sph_to_xyz(coord_Y_tensor.unsqueeze(0)).squeeze(0)

        relative_vect = xyz_X - xyz_Y.unsqueeze(0)   # 3
        euclidean = (torch.norm(relative_vect, dim=-1, keepdim=True) + 1e-8) # 1
        unit_dir = relative_vect / euclidean  # 3

        r_diff = (coord_Y_tensor[0] - coord_X_tensor[:, 0]).unsqueeze(1) # 1
        az_diff = self.angle_diff(coord_X_tensor[:, 1], coord_Y_tensor[1]).unsqueeze(1) # 1
        pl_diff = self.angle_diff(coord_X_tensor[:, 2], coord_Y_tensor[2]).unsqueeze(1) # 1

        # nAAno "token" = 22
        # spatial + 10
        # total features is 32 -> output 22 on linear head

        return torch.cat([bioch_tensor, relative_vect, euclidean,
                          unit_dir, r_diff, az_diff, pl_diff], dim=-1)

    def approx_id(self, vector):
        min_error = float('inf')
        approximate_identity = None
        vector = vector.detach().float().squeeze()
        for aa_id, n_v in self.nAAno_vectors.items():
            n_v_tensor = torch.tensor(n_v, dtype=torch.float32)
            error = torch.norm(vector - n_v_tensor).item()
            if error < min_error:
                min_error = error
                approximate_identity = aa_id

        return approximate_identity


    @staticmethod
    def sph_to_xyz(spherical_vector_batch):
        v = spherical_vector_batch
        r, az, pl = v[:, 0], v[:, 1], v[:, 2]
        x = r * torch.sin(pl) * torch.cos(az)
        y = r * torch.sin(pl) * torch.sin(az)
        z = r * torch.cos(pl)
        return torch.stack([x, y, z], dim=-1)

    @staticmethod
    def angle_diff(aX, aY):
        return (torch.cos(aX) - torch.cos(aY))**2 + (torch.sin(aX) - torch.sin(aY))**2

def encoder_check(verbose=True):
    module = NAAnoEng(max_angstroms=42, block_size=42, verbose=False)
    module.initialize()
    for aa_code, aa_vect in module.nAAno_vectors.items():
        print(f"{aa_code} -- {aa_vect}")
    print(f"{module.n_features()} features")

    # check decoder and encoder
    for aa in AA_IDS:
        aa_str = aa
        aa_emb = module.get_nAAnovector(aa)
        if aa_str == module.get_aa_id(aa_emb):
            if verbose:
                print(f"{aa_str}: str <-> vect aligned")
        else:
            raise ValueError(f"Ensure {aa} in nAAno_library is up to date")
encoder_check()  # note: all good

A -- [9.0, 0, 6.02, 4.1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14.6, 8.7, 7.1, 10.0]
C -- [12.0, 0, 5.02, 4.9, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 9.3, 13.700000000000001, 8.4, 9.7]
D -- [13.0, -5, 2.98, -5.5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9.6, 5.4, 13.100000000000001, 8.7]
E -- [15.0, -5, 3.22, -3.1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14.6, 6.5, 8.5, 8.9]
F -- [17.0, 0, 5.48, 10, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 10.4, 14.399999999999999, 7.0, 8.8]
G -- [8.0, 0, 5.97, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4.2, 3.5999999999999996, 18.5, 9.6]
H -- [16.0, 0, 7.59, 0.8, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8.5, 11.6, 10.2, 8.9]
I -- [13.0, 0, 5.98, 10.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 8.6, 18.5, 5.8999999999999995, 7.6]
K -- [15.0, 5, 9.47, -2.3000000000000003, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11.200000000000001, 8.6, 9.9, 12.4]
L -- [13.0, 0, 5.98, 9.7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 13.200000000000001, 11.5, 6.5, 7.6]
M -- [15.0, 0, 5.

In [5]:
from torch.utils.data import Dataset
import pyarrow.parquet as pq

# note: all coordinates are now on the spherical system
# doesn't need to predict an "end" since that's covered by Skeleton
# all it has to do is predict the next valid suitable biochemical environment

# USE THIS FOR WEBSITE TRAINING

class NAAnoDataset(Dataset):
    def __init__(self, pointer_path, smiles_molfp, pdb_radial,
                 block_size, max_angstroms):
        self.naano_module = NAAnoEng(max_angstroms=max_angstroms,
                                     block_size=block_size,
                                     verbose=False)
        self.naano_module.initialize()

        self.pointer = pq.ParquetFile(pointer_path)
        self.table = pq.read_table(pointer_path, memory_map=True)

        self.smiles_col = self.table.column("SMILES").combine_chunks()
        self.pdb_col = self.table.column("PDB_HIT").combine_chunks()
        self.window_col = self.table.column("WINDOW_IDX").combine_chunks()

        self.smiles_molfp = smiles_molfp
        self.pdb_radial = pdb_radial

        self.block_size = block_size
        self.max_angstroms = max_angstroms

    def __len__(self):
        return len(self.table)

    def __getitem__(self, idx):
        smiles = self.smiles_col[idx].as_py()
        pdb_id = self.pdb_col[idx].as_py()
        target_idx = self.window_col[idx].as_py()

        # context
        molecular_fingerprint = self.smiles_molfp.loc[smiles, "map4_fp"]
        radial_sequence = self.pdb_radial.loc[pdb_id, "radial_sequence"]
        nAAno_X, nAAno_Y = self.naano_XY(radial_sequence, target_idx)

        return (molecular_fingerprint,
                nAAno_X,  # we're already receiving it as a tensor
                torch.tensor(nAAno_Y, dtype=torch.float32))


    def naano_XY(self, radial_sequence, target_idx):
        """
        ok think slightly harder about this
        - we want the relative vector to the target coordinate
        - get context first, record target Y when found and then break
        - calculate relative coordinates and append to biochemical vectors after
        :param radial_sequence:
        :param target_idx:
        :return:
        """
        r_pad = int(self.max_angstroms)
        az_pad = 0
        pl_pad = 0

        coord_context = [[r_pad, az_pad, pl_pad] for _ in range(self.block_size)]
        bioch_context = [self.naano_module.get_nAAnovector("VOID") for _ in range(self.block_size)]

        coord_Y = None  # what we use to calculate the relative vectors
        naano_Y = None  # this is what our model needs to predict

        for idx in range(len(radial_sequence)):
            aa_id = radial_sequence[idx][0][0]
            nAAno_token = self.naano_module.get_nAAnovector(aa_id)  # biochemical vector
            coords = radial_sequence[idx][1]  # convert back to [x, y, z]

            if idx == target_idx:
                coord_Y = coords
                naano_Y = nAAno_token
            else:
                coord_context = coord_context[1:]+[coords]
                bioch_context = bioch_context[1:]+[nAAno_token]

        # create final context matrix
        naano_X = self.naano_module.get_nAAno_X(coord_context, bioch_context, coord_Y)

        return naano_X, naano_Y

    # 22 biochemical features + 7 augmented relative spatial features
    # 29 total -> 29 in embedding -> 22 linear head output

In [21]:
n_nAAno_features = 21
n_spatial_features = 10

radial_resolution = 100
max_angstroms = 33

block_size = 40     # make this 75
batch_size = 64    # make ths 128
n_embd = 256        # 512
n_head = 4          # 8
n_layers = 2        # 6
map4_res = 1024
dropout=0.15
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_epochs = 1        # 3
learning_rate = 1e-4


# training dataset
training_dataset = NAAnoDataset(pointer_path=train_pointers_path,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms,
                                )
# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

torch.manual_seed(311104)

# test / validation set
test_set = NAAnoDataset(pointer_path=test_pointers_path,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms,
                        )# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True
)

In [28]:
# Generates a suitable biochemical environment for a SMILES given nAAnoVector context
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from src.nano_maker.modules.nAAno.naanoeng import NAAnoEng

# NAANOBOT MODEL
class NAAnoBot(nn.Module):
    def __init__(self, n_embd, n_head, n_layers, block_size,
                 map4_res, max_angstroms,
                 n_nAAno_features, n_spatial_features,
                 dropout):
        super().__init__()
        self.block_size = block_size
        self.map4_res = map4_res
        self.n_nAAno_features = n_nAAno_features
        self.n_spatial_features = n_spatial_features
        total_features = self.n_nAAno_features + self.n_spatial_features

        self.max_angstroms = max_angstroms
        self.naano_module = NAAnoEng(max_angstroms=max_angstroms,
                                     block_size=block_size,
                                     verbose=False)
        self.naano_module.initialize()

        # layers + architecture
        self.nAAno_project = nn.Linear(total_features, n_embd)  # feed nAAno feature vector
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.mol_encoder = MolecularEncoder(n_embd, map4_res, dropout)

        self.stacks = nn.ModuleList([Stack(n_embd, n_head, block_size, dropout) for _ in range(n_layers)])

        # OUTPUT HEAD -> outputs feature vector
        self.ln_f = nn.LayerNorm(n_embd)
        self.nAAno_head = nn.Linear(n_embd, n_nAAno_features)


    def forward(self, nAAno_context, map4_enc, targets=None):
        """
        nAAno_context will be [n_batch, block_size, n_features + 3]
        targets is [n_batch, 1, n_features]
        map4_enc -> unsure how I'm going to encode this
        """
        B, T, C = nAAno_context.shape
        nAAno_emb = self.nAAno_project(nAAno_context.float())
        pos_emb = self.pos_emb(torch.arange(T, device=nAAno_context.device))
        x = nAAno_emb + pos_emb

        mol_emb = self.mol_encoder(map4_enc.float()).unsqueeze(1)

        for stack in self.stacks:
            x = stack(x, mol_emb)

        output = self.nAAno_head(self.ln_f(x[:, -1, :]))

        loss = None
        if targets is not None:
            loss = F.mse_loss(output, targets)

        return output, loss


    def generate(self, map4_enc, sph_coordinates):
        """
        Feed it a list of spherical coordinates, and have them converted to what is done in
        :param map4_enc:
        :param sph_coordinates:
        :return:
        """
        map4_enc = map4_enc.to(next(self.parameters()).device)

        bioch_context = [self.naano_module.get_nAAnovector("VOID") for _ in range(self.block_size)]
        coord_context = [[self.max_angstroms, 0, 0] for _ in range(self.block_size)]

        aa_order = []

        for coordinate in sph_coordinates:   # go through spherical coordinates 1 by 1

            # converts to tensors internally
            naano_X = self.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0)

            output, _ = self.forward(naano_X, map4_enc)
            aa_id = self.naano_module.approx_id(output)

            aa_order.append(aa_id)

            next_nAAnovector = self.naano_module.get_nAAnovector(aa_id)
            bioch_context = bioch_context[1:] + [next_nAAnovector]
            coord_context = coord_context[1:] + [coordinate.tolist() if torch.is_tensor(coordinate) else coordinate]


        return aa_order


# most of the stuff below is from the nanoGPT file except the nanogpt and molecular encoder
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

        # added in a causal mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute affinities
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5
        attn_wts = attn_wts.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighed aggregation
        values = self.value(x)
        output = attn_wts @ values
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coordinate_emb, molecular_emb):
        q = self.query(coordinate_emb)
        k = self.key(molecular_emb)
        v = self.value(molecular_emb)

        head_size = q.shape[-1]

        c_attn = q @ k.transpose(-2, -1) * head_size**-0.5
        c_attn = F.softmax(c_attn, dim=-1)
        c_attn = self.dropout(c_attn)

        return c_attn @ v

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,
                                                       head_size,
                                                       dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coord_emb, mol_emb):
        out = torch.cat([h(coord_emb, mol_emb) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class FeedForwardNet(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Stack(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.satt_head = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForwardNet(n_embd, dropout)
        self.catt_head = MultiHeadCrossAttention(n_embd, n_head, dropout)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, mol_emb):
        x = x + self.satt_head(self.ln1(x))
        x = x + self.catt_head(self.ln2(x), mol_emb)
        x = x + self.ffwd(self.ln3(x))
        return x


# previously was way too simple of an MLP, added in more layers
# consider upgrading to use ChemBERTa and save embeddings as vectors to feed in to cross-attention
class MolecularEncoder(nn.Module):
    def __init__(self, n_embd, map4_res, dropout):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Linear(map4_res, map4_res),
            nn.LayerNorm(map4_res),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.check1 = nn.Linear(map4_res, map4_res // 2)

        self.block2 = nn.Sequential(
            nn.Linear(map4_res // 2, map4_res // 2),
            nn.LayerNorm(map4_res // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.check2 = nn.Linear(map4_res // 2, n_embd)
        self.residual1 = nn.Linear(map4_res, map4_res // 2)

        self.block3 = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.LayerNorm(n_embd),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.residual2 = nn.Linear(map4_res // 2, n_embd)


        self.output = nn.LayerNorm(n_embd)

    def forward(self, x):
        x1 = self.block1(x) + x
        x1_down = self.check1(x1)

        x2 = self.block2(x1_down) + self.residual1(x1)
        x2_down = self.check2(x2)

        x3 = self.block3(x2_down) + self.residual2(x2)

        return self.output(x3)

In [29]:
model = NAAnoBot(n_embd=n_embd, n_head=n_head, n_layers=n_layers,
                 block_size=block_size, map4_res=map4_res, max_angstroms=max_angstroms,
                 n_nAAno_features=n_nAAno_features, n_spatial_features=n_spatial_features,
                 dropout=dropout).to(device)

In [30]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

NAAnoBot(
  (nAAno_project): Linear(in_features=31, out_features=256, bias=True)
  (pos_emb): Embedding(40, 256)
  (mol_encoder): MolecularEncoder(
    (block1): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.15, inplace=False)
    )
    (check1): Linear(in_features=1024, out_features=512, bias=True)
    (block2): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.15, inplace=False)
    )
    (check2): Linear(in_features=512, out_features=256, bias=True)
    (residual1): Linear(in_features=1024, out_features=512, bias=True)
    (block3): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2):

In [31]:
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=1000
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(loader) * n_epochs,
    eta_min=3e-5
)

lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[1000]
)

In [32]:
@torch.no_grad()
def estimate_loss(model, loader, device, radial_res, max_batches=None):
    model.eval()
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches and batch_idx >= max_batches:
            break

        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        _, loss = model(radial_X, map4_fp, radial_Y)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches


In [33]:
loss_record = []

for epoch in range(n_epochs):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        optimizer.zero_grad()
        pred, loss = model(radial_X, map4_fp, radial_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()

        loss_record.append(loss.item())
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    train_loss = total_loss / len(loader)
    val_loss = estimate_loss(model, test_loader, device, radial_res=radial_resolution, max_batches=100)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'loss_record': loss_record,
    }, DATAPATH / f"naanobot_epoch{epoch+1}.pt")

    print(f"Checkpoint saved → checkpoints/epoch_{epoch+1}.pt")
    print(f"Epoch {epoch+1} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Gap: {val_loss - train_loss:.6f}")


Epoch 1 | Batch 1 | Loss: 34.303486
Epoch 1 | Batch 51 | Loss: 26.565348
Epoch 1 | Batch 101 | Loss: 18.300461


KeyboardInterrupt: 